In [1]:
import pandas as pd
import gzip
import urllib.request
import os

def get_uniprot_mapping(
    url: str, 
    file_path: str, 
    target_type: str = 'Gene_Name', 
    output_col_name: str = 'gene_symbol'
) -> pd.DataFrame:
    """
    UniProt ID 매핑 데이터를 다운로드하고 특정 ID 타입만 필터링하여 반환합니다.
    
    Args:
        url (str): 데이터 다운로드 URL
        file_path (str): 저장할 로컬 파일 경로 (gz 포함)
        target_type (str): 필터링할 ID 타입 (예: 'Gene_Name', 'Ensembl' 등)
        output_col_name (str): 결과 데이터프레임의 타겟 컬럼명 변경값
        
    Returns:
        pd.DataFrame: ['uniprot_id', output_col_name] 컬럼을 가진 데이터프레임
    """
    
    # 1. 파일 다운로드 (없을 경우에만)
    if not os.path.exists(file_path):
        print(f"📥 '{file_path}' 다운로드 중... (시간이 소요됩니다)")
        try:
            urllib.request.urlretrieve(url, file_path)
            print("✅ 다운로드 완료.")
        except Exception as e:
            print(f"❌ 다운로드 실패: {e}")
            return pd.DataFrame()
    else:
        print(f"ℹ️ 기존 파일 '{file_path}'을(를) 사용합니다.")

    # 2. 데이터 로드 및 필터링
    print(f"🔄 '{target_type}' 데이터 처리 중...")
    chunk_size = 100_000
    results = []

    try:
        with gzip.open(file_path, 'rt') as f:
            # 필요한 컬럼만 지정하여 읽기
            reader = pd.read_csv(
                f, 
                sep='\t', 
                header=None, 
                names=['UniProtID', 'Type', 'Value'], 
                chunksize=chunk_size
            )
            
            for chunk in reader:
                # 조건에 맞는 행만 추출 (메모리 절약)
                subset = chunk.loc[chunk['Type'] == target_type, ['UniProtID', 'Value']]
                results.append(subset)
                
    except Exception as e:
        print(f"파일 읽기 오류: {e}")
        return pd.DataFrame()

    # 3. 결과 병합 및 정리
    if results:
        df_final = pd.concat(results, ignore_index=True)
        
        # 컬럼 이름 변경 (사용자가 지정한 이름으로)
        df_final.columns = ['uniprot_id', output_col_name]
        
        # 중복 제거
        df_final.drop_duplicates(inplace=True)
        
        print(f" 처리 완료: {len(df_final)}개 매핑 발견")
        return df_final
    else:
        print("⚠️ 해당 타입의 데이터가 없습니다.")
        return pd.DataFrame()

# ==========================================
# 실행 예시
# ==========================================
# 쥐(Mouse) 데이터 URL
URL = "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/MOUSE_10090_idmapping.dat.gz"
FILE = "MOUSE_10090_idmapping.dat.gz"
#URL = "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/HUMAN_9606_idmapping.dat.gz"
#FILE = "HUMAN_9606_idmapping.dat.gz"

# 1. Gene Symbol 가져오기 (기본 사용)
df_gene = get_uniprot_mapping(URL, FILE, target_type='Gene_Name', output_col_name='gene_symbol')
print(df_gene.head())

df_final.to_csv("uniprot-data/uniprot_gene_mapping_human.csv", index=False)
df_gene.to_csv("uniprot-data/uniprot_gene_mapping_mouse.csv", index=False)

# 2. (응용) Ensembl ID 가져오기 - 함수 재사용!
# df_ensembl = get_uniprot_mapping(URL, FILE, target_type='Ensembl', output_col_name='ensembl_id')
# print(df_ensembl.head())

ℹ️ 기존 파일 'MOUSE_10090_idmapping.dat.gz'을(를) 사용합니다.
🔄 'Gene_Name' 데이터 처리 중...
 처리 완료: 82653개 매핑 발견
  uniprot_id gene_symbol
0     Q60888     Or10d1b
1     Q8VBW9     Or10h28
2     Q60885     Or10p22
3     Q9CQV8       Ywhab
4     P62259       Ywhae


In [3]:
df_final.to_csv("uniprot-data/uniprot_gene_mapping_human.csv", index=False)
df_gene.to_csv("uniprot-data/uniprot_gene_mapping_mouse.csv", index=False)

In [10]:
df_mouse

,UniProtID,Type,Value
1,Q60888,Gene_Name,Or10d1b
46,Q8VBW9,Gene_Name,Or10h28
85,Q60885,Gene_Name,Or10p22
122,Q9CQV8,Gene_Name,Ywhab
209,P62259,Gene_Name,Ywhae
...,...,...,...
2173534,Q4U213,Gene_Name,PTPphi
2173546,Q9CVJ2,Gene_Name,1810058N15Rik
2173581,Q9D449,Gene_Name,Hsfy2
2173596,Q3V1Y1,Gene_Name,4933407I18Rik


In [ ]:
df_mouse = df_mouse.rename(columns={'UniProtID': 'Uniprot_ID'})
df_mouse = df_mouse.rename(columns={'Value': 'Gene_Symbol'})

In [13]:
df_mouse = df_mouse.drop(columns=['Type'])

In [19]:
df_mouse.to_csv("uniprot_gene_mapping_mouse.csv", index=False)

In [4]:
import pandas as pd
import gzip
import urllib.request
import os

# 1. 매핑 함수 정의 (앞서 만든 것과 동일)
def get_uniprot_mapping(url, file_path, target_type, output_col_name, tax_id, organism_name):
    if not os.path.exists(file_path):
        print(f"다운로드 중: {organism_name}...")
        try:
            urllib.request.urlretrieve(url, file_path)
        except Exception as e:
            print(f"❌ 실패: {e}")
            return pd.DataFrame()
    
    print(f"처리 중: {organism_name} ({target_type})...")
    results = []
    try:
        with gzip.open(file_path, 'rt') as f:
            reader = pd.read_csv(f, sep='\t', header=None, names=['UniProtID', 'Type', 'Value'], chunksize=100000)
            for chunk in reader:
                subset = chunk.loc[chunk['Type'] == target_type, ['UniProtID', 'Value']]
                results.append(subset)
    except Exception as e:
        print(f"❌ 읽기 오류: {e}")
        return pd.DataFrame()

    if results:
        df = pd.concat(results, ignore_index=True)
        df.columns = ['uniprot_id', output_col_name]
        df.drop_duplicates(inplace=True)
        
        # [핵심] 구분자 컬럼 추가
        df['tax_id'] = tax_id
        df['organism'] = organism_name
        return df
    return pd.DataFrame()

# ==========================================
# 실행 및 병합 (Merge)
# ==========================================
os.makedirs("uniprot-data", exist_ok=True)

# 1. Human 데이터 가져오기
df_human = get_uniprot_mapping(
    url="https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/HUMAN_9606_idmapping.dat.gz",
    file_path="uniprot-data/HUMAN_9606_idmapping.dat.gz",
    target_type='Gene_Name',
    output_col_name='gene_symbol',
    tax_id=9606,
    organism_name='Homo sapiens'
)

# 2. Mouse 데이터 가져오기
df_mouse = get_uniprot_mapping(
    url="https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/MOUSE_10090_idmapping.dat.gz",
    file_path="uniprot-data/MOUSE_10090_idmapping.dat.gz",
    target_type='Gene_Name',
    output_col_name='gene_symbol',
    tax_id=10090,
    organism_name='Mus musculus'
)

# 3. [핵심] 데이터 하나로 합치기 (Concatenate)
print(f"\n 데이터 병합 중...")
# ignore_index=True: 인덱스를 0부터 새로 매깁니다.
df_integrated = pd.concat([df_human, df_mouse], ignore_index=True)

# 4. 결과 확인 및 저장
print(f"{'='*50}")
print(f"최종 통합 데이터 완료")
print(f"{'='*50}")
print(f"총 행 수: {len(df_integrated):,}개")
print(f"컬럼 목록: {list(df_integrated.columns)}")

# 종별 데이터 개수 확인
print("\n[종별 데이터 분포]")
print(df_integrated['organism'].value_counts())

# 파일 저장
OUTPUT_FILE = "uniprot-data/uniprot_data_mouse_human.csv"
df_integrated.to_csv(OUTPUT_FILE, index=False)
print(f"\n저장 완료: {OUTPUT_FILE}")

# 미리보기
print("\n[데이터 미리보기]")
print(df_integrated.head())  # Human 쪽
print(df_integrated.tail())  # Mouse 쪽

다운로드 중: Homo sapiens...
처리 중: Homo sapiens (Gene_Name)...
다운로드 중: Mus musculus...
처리 중: Mus musculus (Gene_Name)...

 데이터 병합 중...
최종 통합 데이터 완료
총 행 수: 248,430개
컬럼 목록: ['uniprot_id', 'gene_symbol', 'tax_id', 'organism']

[종별 데이터 분포]
organism
Homo sapiens    165777
Mus musculus     82653
Name: count, dtype: int64

저장 완료: uniprot-data/uniprot_data_mouse_human.csv

[데이터 미리보기]
  uniprot_id gene_symbol  tax_id      organism
0     P31946       YWHAB    9606  Homo sapiens
1     P62258       YWHAE    9606  Homo sapiens
2     Q04917       YWHAH    9606  Homo sapiens
3     P61981       YWHAG    9606  Homo sapiens
4     P31947         SFN    9606  Homo sapiens
       uniprot_id    gene_symbol  tax_id      organism
248425     Q4U213         PTPphi   10090  Mus musculus
248426     Q9CVJ2  1810058N15Rik   10090  Mus musculus
248427     Q9D449          Hsfy2   10090  Mus musculus
248428     Q3V1Y1  4933407I18Rik   10090  Mus musculus
248429     Q3SXC8         Gm5111   10090  Mus musculus
